# Part 5 : Sélection, Validation et Optimisation

**Durée estimée : 3h**

## Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Choisir** le bon algorithme selon le type de problème
2. **Évaluer** de manière robuste avec la validation croisée (K-Fold)
3. **Optimiser** les hyperparamètres avec GridSearchCV et RandomizedSearchCV
4. **Construire** un workflow complet Pipeline + CV + Optimization

---

## Problème Réel : Le dilemme du Data Scientist

Vous avez appris 4 algorithmes (Parts 1-4). Votre manager vous demande :

> "On doit prédire quels clients vont partir (churn). Quel algorithme tu utilises ?"

Vous répondez : "Random Forest, c'est le meilleur !"

Il rétorque : "Comment tu sais ? Tu as testé les autres ? Et avec quels paramètres ?"

**Questions légitimes :**
1. Comment **choisir** le bon algorithme ?
2. Comment **évaluer** de manière fiable (pas juste un split chanceux) ?
3. Comment **trouver** les meilleurs hyperparamètres ?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.datasets import make_classification
from scipy.stats import randint
import warnings
warnings.filterwarnings('ignore')

---

## 5.1 Sélection d'Algorithme : Le Flowchart

### Quel algorithme pour quel problème ?

```
┌───────────────────────────────────────────────────────────────────────────────┐
│                    QUEL ALGORITHME CHOISIR ?                                  │
├───────────────────────────────────────────────────────────────────────────────┤
│                                                                               │
│                        ┌─────────────────────┐                                │
│                        │ Avez-vous des LABELS│                                │
│                        │ (réponses connues)? │                                │
│                        └──────────┬──────────┘                                │
│                           OUI /        \ NON                                  │
│                              /          \                                     │
│              ┌───────────────┐          ┌───────────────┐                     │
│              │ SUPERVISÉ     │          │ NON-SUPERVISÉ │                     │
│              └───────┬───────┘          └───────┬───────┘                     │
│                      │                          │                             │
│                      ▼                          ▼                             │
│    ┌─────────────────────────────┐      ┌─────────────────┐                   │
│    │ La cible est-elle...        │      │ → K-Means       │                   │
│    │ • Un NOMBRE ? → Régression  │      │ → DBSCAN        │                   │
│    │ • Une CATÉGORIE ? → Classif.│      └─────────────────┘                   │
│    └─────────────┬───────────────┘                                            │
│                  │                                                            │
│          ┌──────┴──────┐                                                      │
│          │             │                                                      │
│          ▼             ▼                                                      │
│    ┌──────────┐  ┌───────────────────┐                                        │
│    │RÉGRESSION│  │ CLASSIFICATION    │                                        │
│    └────┬─────┘  └────────┬──────────┘                                        │
│         │                 │                                                   │
│         ▼                 ▼                                                   │
│   Linear Reg.       Logistic Reg.                                            │
│   Random Forest     Random Forest                                             │
│                                                                               │
└───────────────────────────────────────────────────────────────────────────────┘
```

### Tableau comparatif des algorithmes

| Critère | Rég. Linéaire | Rég. Logistique | Arbre | Random Forest | K-Means |
|---------|---------------|-----------------|-------|---------------|--------|
| **Type** | Régression | Classification | Les deux | Les deux | Clustering |
| **Interprétabilité** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| **Performance** | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Vitesse** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Non-linéarité** | ❌ | ❌ | ✅ | ✅ | ✅ |
| **Normalisation** | ⚠️ Parfois | ⚠️ Recommandé | ❌ Non | ❌ Non | ✅ Obligatoire |

### La règle d'or : Commencer Simple

```
1. DummyClassifier/Regressor (baseline triviale)
   │
   ▼ (battre cette baseline)
2. Régression Logistique / Linéaire
   │
   ▼ (si pas assez performant)
3. Random Forest
   │
   ▼ (si besoin de +2-3%)
4. Gradient Boosting (XGBoost)
```

**Pourquoi ?** Un modèle simple qui fonctionne > Un modèle complexe jamais livré.

---

## 5.2 Validation Croisée : Évaluer de Manière Robuste

### Le problème du split unique

Avec un seul train/test split, votre score dépend de **quels exemples** tombent dans test.

In [ ]:
# Créer un dataset
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           random_state=42)

# Démontrer la variabilité du split
model = LogisticRegression(max_iter=1000, random_state=42)

scores = []
for rs in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)
    model.fit(X_train, y_train)
    scores.append(model.score(X_test, y_test))

print("Variabilité selon random_state")
print("═" * 50)
print(f"Min: {min(scores):.1%} | Max: {max(scores):.1%} | Écart: {max(scores)-min(scores):.1%}")
print("\n→ Un seul split n'est pas fiable !")

### K-Fold Cross-Validation : La solution

```
┌─────────────────────────────────────────────────────────────────────┐
│              K-FOLD CROSS-VALIDATION (K=5)                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Fold 1 : [TEST]  [TRAIN] [TRAIN] [TRAIN] [TRAIN] → Score 1       │
│   Fold 2 : [TRAIN] [TEST]  [TRAIN] [TRAIN] [TRAIN] → Score 2       │
│   Fold 3 : [TRAIN] [TRAIN] [TEST]  [TRAIN] [TRAIN] → Score 3       │
│   Fold 4 : [TRAIN] [TRAIN] [TRAIN] [TEST]  [TRAIN] → Score 4       │
│   Fold 5 : [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]  → Score 5       │
│                                                                     │
│   Score Final = Moyenne ± Écart-type                                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Cross-validation avec cross_val_score
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000, random_state=42)
scores_cv = cross_val_score(model, X, y, cv=5)

print("5-Fold Cross-Validation")
print("═" * 50)
for i, score in enumerate(scores_cv, 1):
    print(f"  Fold {i} : {score:.4f}")

print(f"\n  Moyenne : {scores_cv.mean():.4f}")
print(f"  Écart-type : {scores_cv.std():.4f}")
print(f"\n→ Format standard : {scores_cv.mean():.2%} ± {scores_cv.std():.2%}")

### Stratified K-Fold : Préserver les proportions

Pour les classes déséquilibrées, utiliser **StratifiedKFold** qui garantit la même proportion de classes dans chaque fold.

**Note :** `cross_val_score` utilise automatiquement StratifiedKFold pour les classifieurs.

In [ ]:
# Comparer plusieurs modèles avec CV
modeles = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

print("Comparaison de modèles (5-Fold CV)")
print("═" * 60)

resultats = []
for nom, model in modeles.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    resultats.append({'Modèle': nom, 'Moyenne': scores.mean(), 'Std': scores.std()})
    print(f"{nom:25s} : {scores.mean():.2%} ± {scores.std():.2%}")

# Meilleur
meilleur = max(resultats, key=lambda x: x['Moyenne'])
print(f"\n🏆 Meilleur : {meilleur['Modèle']}")

---

## 5.3 Optimisation des Hyperparamètres

### Rappel : Paramètres vs Hyperparamètres

| | Paramètres | Hyperparamètres |
|---|---|---|
| Appris par | `fit()` | Vous (AVANT fit) |
| Exemples | Coefficients, seuils | max_depth, n_estimators, C |
| Choix | Automatique | À optimiser |

### GridSearchCV : Tester TOUTES les combinaisons

```
┌─────────────────────────────────────────────────────────────────────┐
│              GRIDSEARCHCV                                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Grille :                                                          │
│   n_estimators = [50, 100, 200]                                     │
│   max_depth = [5, 10, None]                                         │
│                                                                     │
│   → 3 × 3 = 9 combinaisons                                          │
│   → Avec CV=5 : 9 × 5 = 45 entraînements                           │
│                                                                     │
│   GridSearchCV teste TOUT et garde le meilleur.                     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Préparer les données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Définir le modèle et la grille
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

n_comb = 3 * 3 * 2
print(f"Combinaisons à tester : {n_comb}")
print(f"Avec CV=5 : {n_comb * 5} entraînements")

In [ ]:
# Lancer GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("\nLancement de GridSearchCV...")
grid_search.fit(X_train, y_train)

In [ ]:
# Résultats
print("\nRésultats GridSearchCV")
print("═" * 55)

print("\n🏆 Meilleurs hyperparamètres :")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {grid_search.best_score_:.4f}")

# Test final
test_score = grid_search.best_estimator_.score(X_test, y_test)
print(f"🎯 Score TEST (jamais vu) : {test_score:.4f}")

### RandomizedSearchCV : Quand la grille est trop grande

Au lieu de TOUT tester, échantillonner N combinaisons aléatoires.

In [ ]:
# Distributions continues
param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=30,  # Seulement 30 combinaisons
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Lancement de RandomizedSearchCV (30 itérations)...")
random_search.fit(X_train, y_train)

In [ ]:
# Résultats
print("\nRésultats RandomizedSearchCV")
print("═" * 55)

print("\n🏆 Meilleurs hyperparamètres :")
for param, value in random_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {random_search.best_score_:.4f}")
print(f"🎯 Score TEST : {random_search.best_estimator_.score(X_test, y_test):.4f}")

### GridSearch vs RandomizedSearch

| | GridSearchCV | RandomizedSearchCV |
|---|---|---|
| Explore | TOUTES les combinaisons | N combinaisons aléatoires |
| Avantage | Exhaustif | Rapide, espace continu |
| Inconvénient | Lent si beaucoup de params | Peut manquer l'optimal |
| Quand l'utiliser | Peu de params, valeurs discrètes | Beaucoup de params, exploration |

---

## 5.4 Workflow Complet : Pipeline + CV + Optimization

### Éviter le data leakage

**Problème :** Si vous faites le preprocessing AVANT GridSearchCV, le scaler a "vu" les données de validation = data leakage !

**Solution :** Mettre le preprocessing DANS un Pipeline.

In [ ]:
# Pipeline complet
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Grille pour le pipeline
# Note : préfixe 'classifier__' pour les params du RF
param_grid_pipeline = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None],
}

print("Pipeline créé :")
print(pipeline)

In [ ]:
# GridSearchCV sur le pipeline
grid_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid_pipeline,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("\nLancement de GridSearchCV sur Pipeline...")
grid_pipeline.fit(X_train, y_train)

In [ ]:
# Résultats finaux
print("\nRésultats Pipeline + GridSearchCV")
print("═" * 55)

print("\n🏆 Meilleurs hyperparamètres :")
for param, value in grid_pipeline.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {grid_pipeline.best_score_:.4f}")
print(f"🎯 Score TEST : {grid_pipeline.best_estimator_.score(X_test, y_test):.4f}")

print("\n✅ Le preprocessing est fait DANS chaque fold = pas de leakage !")

### Workflow recommandé

```
┌─────────────────────────────────────────────────────────────────────┐
│           WORKFLOW COMPLET                                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   1. SÉPARER train / test final                                     │
│      X_train, X_test = train_test_split(...)                        │
│                                                                     │
│   2. CRÉER un Pipeline (preprocessing + modèle)                     │
│      pipeline = Pipeline([('scaler', ...), ('model', ...)])         │
│                                                                     │
│   3. DÉFINIR la grille d'hyperparamètres                           │
│      param_grid = {'model__param': [val1, val2]}                   │
│                                                                     │
│   4. LANCER GridSearchCV                                           │
│      grid = GridSearchCV(pipeline, param_grid, cv=5)               │
│      grid.fit(X_train, y_train)                                    │
│                                                                     │
│   5. ÉVALUER sur test final                                        │
│      grid.best_estimator_.score(X_test, y_test)                    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## Exercice Pratique : Algorithm Tournament Complet

Comparez plusieurs algorithmes sur un dataset de churn et trouvez le meilleur modèle optimisé.

In [ ]:
# Dataset Churn
np.random.seed(42)
X_churn, y_churn = make_classification(
    n_samples=2000, n_features=15, n_informative=8,
    weights=[0.85, 0.15], random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)

print("Dataset Churn")
print("═" * 50)
print(f"Train: {len(X_train_c)} | Test: {len(X_test_c)}")
print(f"Taux de churn: {y_churn.mean():.1%}")

### Mission :
1. Comparer 3 modèles avec 5-Fold CV (Logistic Reg, Decision Tree, Random Forest)
2. Optimiser le meilleur avec GridSearchCV
3. Évaluer sur le test set avec F1-score

In [ ]:
# À VOUS DE JOUER !

# 1. Comparer modèles
# ...

# 2. Optimiser le meilleur
# ...

# 3. Évaluer
# ...

In [ ]:
# SOLUTION

# 1. Comparer modèles
print("1️⃣ Comparaison des modèles (5-Fold CV, F1)")
print("═" * 55)

modeles_churn = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

best_model_name = None
best_score = 0

for nom, model in modeles_churn.items():
    scores = cross_val_score(model, X_train_c, y_train_c, cv=5, scoring='f1')
    print(f"{nom:25s} : F1 = {scores.mean():.2%} ± {scores.std():.2%}")
    if scores.mean() > best_score:
        best_score = scores.mean()
        best_model_name = nom

print(f"\n🏆 Meilleur : {best_model_name}")

In [ ]:
# 2. Optimiser Random Forest
print("\n2️⃣ Optimisation de Random Forest")
print("═" * 55)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [5, 10, 15, None],
    'rf__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_rf.fit(X_train_c, y_train_c)

print(f"\n🏆 Meilleurs params: {grid_rf.best_params_}")
print(f"📈 Meilleur F1 CV: {grid_rf.best_score_:.4f}")

In [ ]:
# 3. Évaluer sur test
print("\n3️⃣ Évaluation finale sur TEST")
print("═" * 55)

y_pred_final = grid_rf.best_estimator_.predict(X_test_c)

print(f"\nAccuracy  : {accuracy_score(y_test_c, y_pred_final):.4f}")
print(f"Precision : {precision_score(y_test_c, y_pred_final):.4f}")
print(f"Recall    : {recall_score(y_test_c, y_pred_final):.4f}")
print(f"F1-Score  : {f1_score(y_test_c, y_pred_final):.4f}")

print("\n✅ Modèle optimisé et évalué proprement !")

---

## Récapitulatif

### Structure de cette partie :

| Section | Contenu |
|---------|--------|
| **5.1** | Flowchart de sélection, tableau comparatif |
| **5.2** | K-Fold CV, Stratified K-Fold, cross_val_score |
| **5.3** | GridSearchCV, RandomizedSearchCV |
| **5.4** | Pipeline + CV + Optimization (workflow complet) |

### Code essentiel :

```python
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline

# 1. Validation croisée
scores = cross_val_score(model, X, y, cv=5, scoring='f1')
print(f"{scores.mean():.2%} ± {scores.std():.2%}")

# 2. Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

# 3. GridSearchCV
param_grid = {'model__n_estimators': [50, 100], 'model__max_depth': [5, 10]}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_estimator_.score(X_test, y_test))
```

---

## Félicitations !

Vous avez terminé le **Chapitre 3 : Algorithmes ML et Évaluation**.

Vous maîtrisez maintenant :
- ✅ La régression linéaire (prédire des nombres)
- ✅ La régression logistique (classifier)
- ✅ Les arbres et Random Forest (performance robuste)
- ✅ K-Means (clustering)
- ✅ La sélection d'algorithme, la validation croisée et l'optimisation

**Prochain chapitre :** Réseaux de Neurones (Conceptuel)

---

## Réflexion Métacognitive

1. Pourquoi commencer par un modèle simple avant Random Forest ?
2. Que signifie un écart-type élevé dans les résultats de CV ?
3. Pourquoi le test set final ne doit JAMAIS être utilisé pendant GridSearch ?

---

**Sources :**
- [scikit-learn Cross-Validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [scikit-learn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)